# Kaggle Dataset Analysis

This notebook downloads a dataset directly from Kaggle, loads it with pandas, and runs a starter exploratory analysis.

## 1. Environment

Recommended setup from the project root:

```bash
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name ai-job-impact-analysis --display-name "AI Job Impact Analysis"
.venv/bin/jupyter lab
```

The dataset is downloaded with `kagglehub`. If authentication is required, run KaggleHub's login flow or configure your Kaggle credentials before running the download cell. Downloaded data is stored under `data/kagglehub`, which is ignored by git.

## 2. Imports and Configuration

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MPLCONFIGDIR = PROJECT_ROOT / ".matplotlib"
KAGGLEHUB_CACHE = PROJECT_ROOT / "data" / "kagglehub"

MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
KAGGLEHUB_CACHE.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
os.environ.setdefault("KAGGLEHUB_CACHE", str(KAGGLEHUB_CACHE))

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

KAGGLE_DATASET = "muhammadwaqas023/ai-impact-in-future-on-jobs-market-in-2030"

SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xls", ".json", ".parquet"}

## 3. Download from Kaggle

In [ ]:
dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))

print("Path to dataset files:", dataset_path)

downloaded_files = sorted(
    path for path in dataset_path.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

downloaded_files

## 4. Load Data

In [ ]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported file type: {suffix}")


if not downloaded_files:
    raise FileNotFoundError(f"No supported data files found in {dataset_path}")

# If Kaggle downloads multiple data files, choose the one to analyze here.
DATA_FILE = downloaded_files[0]
print(f"Loading: {DATA_FILE}")

df = load_dataset(DATA_FILE)
df.head()

## 5. Basic Structure

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.info()

In [ ]:
df.describe(include="all").T

## 6. Data Quality Checks

In [ ]:
missing = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: x["missing_count"] / len(df) * 100)
    .sort_values("missing_pct", ascending=False)
)

missing[missing["missing_count"] > 0]

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count:,}")

In [ ]:
unique_counts = df.nunique(dropna=False).sort_values(ascending=False)
unique_counts.to_frame("unique_values")

## 7. Exploratory Analysis

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

In [ ]:
if numeric_cols:
    df[numeric_cols].hist(figsize=(14, 10), bins=30)
    plt.tight_layout()
else:
    print("No numeric columns found.")

In [ ]:
for col in categorical_cols[:10]:
    display(df[col].value_counts(dropna=False).head(20).to_frame("count"))

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    display(corr)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
    ax.set_yticks(range(len(corr.index)), corr.index)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
else:
    print("Need at least two numeric columns for correlation analysis.")

## 8. Focused Questions

Add dataset-specific questions here, then create analysis cells below each question.

- What are the most important trends or patterns?
- Which variables appear related to the target outcome?
- Are there meaningful differences across groups?
- What data quality issues could affect conclusions?

In [ ]:
# Example: replace with a dataset-specific grouping column and metric.
# df.groupby("category_column")["numeric_metric"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False)

## 9. Findings

Summarize the main findings, caveats, and recommended next steps here.